# ZFN Sharding Default Tuning

This notebook is for choosing sensible internal ZFN sharding defaults, not for adding more runtime toggles.

It does four things:

1. Models the current internal configuration surface and the reconstructed historical baseline.
2. Defines a small set of stable candidate default profiles with typed configuration.
3. Benchmarks those profiles through the in-process Python search API.
4. Writes reproducible artifacts that support a code-default change.

For real hg38 runs, keep the genomic annotation explicit in the notebook and pass the Ensembl GTF URL directly rather than relying on inferred defaults.

The benchmark target is wall-clock runtime under exhaustive search, while preserving top-site consistency for `window_stride=1` profiles.

In [ ]:
from __future__ import annotations

import csv
import importlib
import json
import logging
import math
import statistics
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any

import pandas as pd
from Bio.Seq import Seq
from IPython.display import display

import sirnaforge.models.zfn as zfn_models
import sirnaforge.zfn.search as zfn_search

importlib.reload(zfn_models)
importlib.reload(zfn_search)

from sirnaforge.cli import _autotune_zfn_sharding
from sirnaforge.models.zfn import (
    GenomicAnnotationConfig,
    ZFNAlgorithm,
    ZFNDesignParameters,
    ZFNHalfSiteConstraints,
    ZFNShardingConfig,
    ZFNSpacerConstraints,
)
from sirnaforge.zfn.search import ExhaustiveZFNOffTargetSearcher

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path("/home/hovland/sirnaforge/sirnaforge/notebooks")
WORKSPACE_ROOT = NOTEBOOK_DIR.parent
RUN_ROOT = NOTEBOOK_DIR / "zfn_experiment_runs"
ARTIFACT_ROOT = RUN_ROOT / "sharding_tuning"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = RUN_ROOT / "inputs"
INPUT_ROOT.mkdir(parents=True, exist_ok=True)

CCR5_LEFT_HALF_SITE = "GTCATCCTCATC"
CCR5_RIGHT_HALF_SITE = "AAACTGCAAAAG"
BASE_SPACERS = [5, 6]
DEFAULT_CORES_BUDGET = 8
DEFAULT_REPEATS = 3
DEFAULT_EXHAUSTIVE_REPEATS = 1
HG38_ANNOTATION_URL = "https://ftp.ensembl.org/pub/current_gtf/homo_sapiens/Homo_sapiens.GRCh38.115.gtf.gz"
HG38_ANNOTATION = GenomicAnnotationConfig(annotation_reference=HG38_ANNOTATION_URL)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")
print(
    "Current shipped typed defaults:",
    {
        "chunk_size_bp": ZFNShardingConfig().chunk_size_bp,
        "max_workers": ZFNShardingConfig().max_workers,
        "overlap_bp": ZFNShardingConfig().overlap_bp,
    },
)
print(f"Default exhaustive repeats: {DEFAULT_EXHAUSTIVE_REPEATS}")
print(f"Explicit hg38 annotation URL for real runs: {HG38_ANNOTATION_URL}")

## 1. Model Current Configuration Inputs

The current configuration surface is already mostly typed. The parts that matter for default selection are:

- `ZFNShardingConfig` model defaults.
- CLI autotuning in `_autotune_zfn_sharding()`.
- Internal safety caps in `ExhaustiveZFNOffTargetSearcher`.

The goal here is to make those competing defaults visible and benchmarkable without relying on environment-variable branches.

In [ ]:
current_model_default = ZFNShardingConfig()
autotuned_default = _autotune_zfn_sharding(DEFAULT_CORES_BUDGET)

current_surface = pd.DataFrame(
    [
        {
            "stage": "typed_model_default",
            "setting": "chunk_size_bp",
            "value": current_model_default.chunk_size_bp,
            "notes": "Current typed default when no workflow autotuning is applied",
        },
        {
            "stage": "typed_model_default",
            "setting": "max_workers",
            "value": current_model_default.max_workers,
            "notes": "Current typed default when no workflow autotuning is applied",
        },
        {
            "stage": "typed_model_default",
            "setting": "overlap_bp",
            "value": current_model_default.overlap_bp,
            "notes": "Minimum overlap before half-site geometry raises it",
        },
        {
            "stage": "workflow_default_request",
            "setting": "chunk_size_bp_at_8_cores",
            "value": autotuned_default.chunk_size_bp,
            "notes": "Current workflow path for a representative 8-core budget",
        },
        {
            "stage": "workflow_default_request",
            "setting": "max_workers_at_8_cores",
            "value": autotuned_default.max_workers,
            "notes": "Requested shard workers before searcher safety caps apply",
        },
        {
            "stage": "searcher_guardrail",
            "setting": "thread_worker_cap",
            "value": ExhaustiveZFNOffTargetSearcher._DEFAULT_THREAD_WORKER_CAP,
            "notes": "Internal concurrency safeguard for the in-process threaded backend",
        },
        {
            "stage": "searcher_guardrail",
            "setting": "per_worker_memory_gb",
            "value": ExhaustiveZFNOffTargetSearcher._DEFAULT_PER_WORKER_GB,
            "notes": "Internal memory estimate used to bound effective concurrency",
        },
        {
            "stage": "searcher_guardrail",
            "setting": "memory_reserve_gb",
            "value": ExhaustiveZFNOffTargetSearcher._DEFAULT_MEMORY_RESERVE_GB,
            "notes": "Internal host memory reserve applied by the searcher",
        },
    ]
)

reconstructed_history = pd.DataFrame(
    [
        {
            "profile": "legacy_model_default_pre_2026_03",
            "chunk_size_bp": 20_000_000,
            "max_workers": 1,
            "window_stride": 1,
            "why_it_matters": "Previous typed baseline kept for historical comparison",
        },
        {
            "profile": "current_shipped_default",
            "chunk_size_bp": int(current_model_default.chunk_size_bp),
            "max_workers": int(current_model_default.max_workers),
            "window_stride": 1,
            "why_it_matters": "Current shipped typed default in the model layer",
        },
        {
            "profile": "current_workflow_default_8_cores",
            "chunk_size_bp": int(autotuned_default.chunk_size_bp),
            "max_workers": int(autotuned_default.max_workers),
            "window_stride": 1,
            "why_it_matters": "Current practical default request before internal guardrails cap execution",
        },
    ]
)

print("Current internal default resolution surface")
display(current_surface)
print("\nReconstructed historical comparison anchors")
display(reconstructed_history)

## 2. Define Typed Internal Defaults

The candidate set should stay small and interpretable. These profiles represent plausible code-default choices rather than every possible runtime combination.

In [ ]:
@dataclass(frozen=True)
class ProfileSpec:
    profile_id: str
    chunk_size_bp: int
    max_workers: int
    overlap_bp: int = 50
    window_stride: int = 1
    enabled: bool = True
    notes: str = ""


@dataclass(frozen=True)
class ThresholdSpec:
    min_speedup_vs_baseline: float = 0.15
    max_relative_spread: float = 0.20


LEGACY_MODEL_PROFILE = ProfileSpec(
    profile_id="legacy_model_default",
    chunk_size_bp=20_000_000,
    max_workers=1,
    overlap_bp=50,
    window_stride=1,
    notes="Previous typed model default retained for historical comparison",
)

CURRENT_SHIPPED_PROFILE = ProfileSpec(
    profile_id="current_shipped_default",
    chunk_size_bp=int(ZFNShardingConfig().chunk_size_bp),
    max_workers=int(ZFNShardingConfig().max_workers),
    overlap_bp=int(ZFNShardingConfig().overlap_bp),
    window_stride=1,
    notes="Current shipped typed model default",
)

COMPARISON_BASELINE_PROFILE = CURRENT_SHIPPED_PROFILE

AUTO_8 = _autotune_zfn_sharding(DEFAULT_CORES_BUDGET)
CURRENT_AUTOTUNE_PROFILE = ProfileSpec(
    profile_id="current_autotune_8core",
    chunk_size_bp=int(AUTO_8.chunk_size_bp),
    max_workers=int(AUTO_8.max_workers),
    overlap_bp=int(AUTO_8.overlap_bp),
    window_stride=1,
    notes="Current CLI autotuned shape at 8 cores",
)

CANDIDATE_PROFILES = [
    LEGACY_MODEL_PROFILE,
    CURRENT_SHIPPED_PROFILE,
    CURRENT_AUTOTUNE_PROFILE,
    ProfileSpec(
        profile_id="candidate_12m_1w",
        chunk_size_bp=12_000_000,
        max_workers=1,
        overlap_bp=50,
        window_stride=1,
        notes="Current chunking with serial execution to isolate worker-count effects",
    ),
    ProfileSpec(
        profile_id="candidate_8m_2w",
        chunk_size_bp=8_000_000,
        max_workers=2,
        overlap_bp=50,
        window_stride=1,
        notes="Smaller shards while staying within the shipped two-worker threaded cap",
    ),
    ProfileSpec(
        profile_id="candidate_16m_2w",
        chunk_size_bp=16_000_000,
        max_workers=2,
        overlap_bp=50,
        window_stride=1,
        notes="Larger shard sizing with the same worker budget as the shipped default",
    ),
    ProfileSpec(
        profile_id="exploratory_stride2_12m_2w",
        chunk_size_bp=12_000_000,
        max_workers=2,
        overlap_bp=50,
        window_stride=2,
        notes="Speed exploration only; not exhaustive and not eligible as the default",
    ),
]

DEFAULT_BENCHMARK_PROFILE_IDS = [
    LEGACY_MODEL_PROFILE.profile_id,
    CURRENT_SHIPPED_PROFILE.profile_id,
    CURRENT_AUTOTUNE_PROFILE.profile_id,
    "candidate_8m_2w",
    "candidate_16m_2w",
]

pd.DataFrame(asdict(profile) for profile in CANDIDATE_PROFILES)

## 3. Implement Configuration Resolution Order

Resolution should be predictable:

1. Start from a typed baseline profile.
2. Apply explicit profile fields.
3. Build `ZFNDesignParameters` directly.
4. Let the searcher enforce only its internal safety caps.

This keeps benchmarking focused on the code defaults we might actually ship.

In [ ]:
def resolve_profile(
    profile: ProfileSpec,
    *,
    baseline: ProfileSpec = COMPARISON_BASELINE_PROFILE,
) -> ProfileSpec:
    """Resolve one candidate profile against the comparison baseline."""
    merged = replace(
        baseline,
        profile_id=profile.profile_id,
        chunk_size_bp=profile.chunk_size_bp,
        max_workers=profile.max_workers,
        overlap_bp=profile.overlap_bp,
        window_stride=profile.window_stride,
        enabled=profile.enabled,
        notes=profile.notes or baseline.notes,
    )
    return merged


def build_params(profile: ProfileSpec, search_space_fasta: Path) -> ZFNDesignParameters:
    """Construct typed search parameters directly from a profile."""
    resolved = resolve_profile(profile)
    half_site_constraints = ZFNHalfSiteConstraints(max_mismatches=3, window_stride=resolved.window_stride)
    spacer_constraints = ZFNSpacerConstraints(allowed_spacer_lengths=BASE_SPACERS)
    sharding = ZFNShardingConfig(
        enabled=resolved.enabled,
        chunk_size_bp=resolved.chunk_size_bp,
        overlap_bp=resolved.overlap_bp,
        chromosomes=[],
        max_workers=resolved.max_workers,
    )
    return ZFNDesignParameters(
        search_space_reference=None,
        search_space_fasta=str(search_space_fasta),
        left_half_site=CCR5_LEFT_HALF_SITE,
        right_half_site=CCR5_RIGHT_HALF_SITE,
        half_site_constraints=half_site_constraints,
        spacer_constraints=spacer_constraints,
        algorithm=ZFNAlgorithm.ZFN_V2,
        sharding=sharding,
    )


pd.DataFrame(asdict(resolve_profile(profile)) for profile in CANDIDATE_PROFILES)

## 4. Replace Environment Variable Toggles with Stable Options

The benchmark surface is intentionally narrow:

- `chunk_size_bp`
- `max_workers`
- `overlap_bp`
- `window_stride`

Everything else stays on the ordinary typed defaults unless there is a strong reason to promote it into the benchmark.

In [ ]:
def required_overlap_bp(params: ZFNDesignParameters) -> int:
    max_spacer = max(params.spacer_constraints.allowed_spacer_lengths)
    return max(
        50,
        max(len(params.left_half_site), params.half_site_constraints.max_len)
        + max_spacer
        + max(len(params.right_half_site), params.half_site_constraints.max_len),
    )


def normalize_profile(profile: ProfileSpec) -> ProfileSpec:
    if profile.chunk_size_bp < 1:
        raise ValueError(f"chunk_size_bp must be >= 1 for {profile.profile_id}")
    if profile.max_workers < 1:
        raise ValueError(f"max_workers must be >= 1 for {profile.profile_id}")
    if profile.window_stride < 1:
        raise ValueError(f"window_stride must be >= 1 for {profile.profile_id}")
    params = build_params(profile, INPUT_ROOT / "placeholder.fa")
    normalized_overlap = max(profile.overlap_bp, required_overlap_bp(params))
    return replace(profile, overlap_bp=normalized_overlap)


stable_profile_df = pd.DataFrame(asdict(normalize_profile(profile)) for profile in CANDIDATE_PROFILES)
stable_profile_df

## 5. Add Validation and Normalization

This section builds the synthetic benchmark input and validates that profile normalization is deterministic before any expensive search runs start.

In [ ]:
fixture_csv = WORKSPACE_ROOT / "tests" / "unit" / "data" / "zfn" / "ccr5_s10_visible_rows.csv"
if not fixture_csv.exists():
    raise FileNotFoundError(f"Could not find fixture: {fixture_csv}")

with fixture_csv.open(newline="", encoding="utf-8") as fh:
    fixture_rows = list(csv.DictReader(fh))


def fixture_row(gene: str) -> dict[str, str]:
    return next(row for row in fixture_rows if row.get("Closest gene") == gene)


def site_seq(plus: str, minus: str, spacer_len: int = 5) -> str:
    return plus.upper() + ("A" * spacer_len) + str(Seq(minus.upper()).reverse_complement())


def _benchmark_unit_sequence() -> str:
    ccr5_row = fixture_row("CCR5")
    csnk1g3_row = fixture_row("CSNK1G3")
    tmod1_row = fixture_row("TMOD1")
    sites = [
        site_seq(ccr5_row["(+) half-site"], ccr5_row["(−) half-site"], 5),
        site_seq(csnk1g3_row["(+) half-site"], csnk1g3_row["(−) half-site"], 5),
        site_seq(tmod1_row["(+) half-site"], tmod1_row["(−) half-site"], 5),
    ]
    return "N" * 50 + ("N" * 100).join(sites) + "N" * 50


def _write_fasta_records(fasta_path: Path, records: list[tuple[str, str]], line_width: int = 80) -> Path:
    with fasta_path.open("w", encoding="utf-8") as fh:
        for chrom, sequence in records:
            fh.write(f">{chrom}\n")
            for start in range(0, len(sequence), line_width):
                fh.write(sequence[start : start + line_width])
                fh.write("\n")
    return fasta_path


def build_synthetic_search_space() -> Path:
    fasta_path = INPUT_ROOT / "ccr5_synthetic_genome.fa"
    genome_seq = _benchmark_unit_sequence()
    return _write_fasta_records(fasta_path, [("chr_synthetic", genome_seq)])


def build_sharded_tuning_search_space(target_bp: int = 48_000_000) -> Path:
    fasta_path = INPUT_ROOT / f"ccr5_synthetic_sharded_{target_bp // 1_000_000}mb.fa"
    unit = _benchmark_unit_sequence()
    repeats = math.ceil(target_bp / len(unit))
    genome_seq = (unit * repeats)[:target_bp]
    return _write_fasta_records(fasta_path, [("chr_synthetic_large", genome_seq)])


def build_diverse_tuning_search_space(contig_bp: int = 18_000_000) -> Path:
    fasta_path = INPUT_ROOT / f"ccr5_synthetic_diverse_{(4 * contig_bp) // 1_000_000}mb.fa"
    unit = _benchmark_unit_sequence()
    rev_unit = str(Seq(unit).reverse_complement())
    gc_rich = ("GCGCGTAT" * math.ceil(contig_bp / 8))[:contig_bp]
    at_rich = ("ATATTAAT" * math.ceil(contig_bp / 8))[:contig_bp]
    mixed = ((unit + "N" * 97 + rev_unit + "C" * 83) * math.ceil(contig_bp / (len(unit) * 2 + 180)))[:contig_bp]
    decoy = ("ACGT" * 30) + unit[:64] + ("TTGC" * 25) + rev_unit[:64]
    decoy_sparse = ((decoy + "N" * 401) * math.ceil(contig_bp / (len(decoy) + 401)))[:contig_bp]
    records = [
        ("chr_gc_rich", gc_rich),
        ("chr_at_rich", at_rich),
        ("chr_mixed_dense", mixed),
        ("chr_decoy_sparse", decoy_sparse),
    ]
    return _write_fasta_records(fasta_path, records)


def _patterned_sequence(length_bp: int, variant: str) -> str:
    unit = _benchmark_unit_sequence()
    rev_unit = str(Seq(unit).reverse_complement())
    if variant == "mixed":
        seed = unit + ("N" * 97) + rev_unit + ("C" * 83)
    elif variant == "gc":
        seed = "GCGCGTAT"
    elif variant == "at":
        seed = "ATATTAAT"
    elif variant == "decoy":
        seed = ("ACGT" * 30) + unit[:64] + ("TTGC" * 25) + rev_unit[:64] + ("N" * 401)
    elif variant == "repeat_unit":
        seed = unit
    else:
        raise ValueError(f"Unknown sequence variant: {variant}")
    return (seed * math.ceil(length_bp / len(seed)))[:length_bp]


def build_load_case_search_space(case_name: str, contig_specs: list[tuple[str, int, str]]) -> Path:
    fasta_path = INPUT_ROOT / f"{case_name}.fa"
    records = [(chrom_name, _patterned_sequence(length_bp, variant)) for chrom_name, length_bp, variant in contig_specs]
    return _write_fasta_records(fasta_path, records)


def fasta_total_bp(fasta_path: Path) -> int:
    return sum(
        len(line.strip()) for line in fasta_path.read_text(encoding="utf-8").splitlines() if not line.startswith(">")
    )


SYNTHETIC_FASTA = build_synthetic_search_space()
SMALL_SINGLE_FASTA = build_load_case_search_space(
    "ccr5_load_small_single_9mb",
    [("chr_small", 9_000_000, "mixed")],
)
DUAL_CONTIG_FASTA = build_load_case_search_space(
    "ccr5_load_dual_contig_24mb",
    [
        ("chr_left", 12_000_000, "mixed"),
        ("chr_right", 12_000_000, "decoy"),
    ],
)
SKEWED_CONTIG_FASTA = build_load_case_search_space(
    "ccr5_load_skewed_46mb",
    [
        ("chr_major", 30_000_000, "repeat_unit"),
        ("chr_gc", 10_000_000, "gc"),
        ("chr_at", 4_000_000, "at"),
        ("chr_sparse", 2_000_000, "decoy"),
    ],
)
SHARDED_TUNING_FASTA = build_sharded_tuning_search_space()
DIVERSE_TUNING_FASTA = build_diverse_tuning_search_space()

print(f"Smoke FASTA: {SYNTHETIC_FASTA}")
print(f"Smoke size (bp): {fasta_total_bp(SYNTHETIC_FASTA)}")
print(f"Small single-contig FASTA: {SMALL_SINGLE_FASTA}")
print(f"Small single-contig size (bp): {fasta_total_bp(SMALL_SINGLE_FASTA)}")
print(f"Dual-contig FASTA: {DUAL_CONTIG_FASTA}")
print(f"Dual-contig size (bp): {fasta_total_bp(DUAL_CONTIG_FASTA)}")
print(f"Skewed-contig FASTA: {SKEWED_CONTIG_FASTA}")
print(f"Skewed-contig size (bp): {fasta_total_bp(SKEWED_CONTIG_FASTA)}")
print(f"Sharded tuning FASTA: {SHARDED_TUNING_FASTA}")
print(f"Sharded tuning size (bp): {fasta_total_bp(SHARDED_TUNING_FASTA)}")
print(f"Diverse tuning FASTA: {DIVERSE_TUNING_FASTA}")
print(f"Diverse tuning size (bp): {fasta_total_bp(DIVERSE_TUNING_FASTA)}")

normalized_profiles = [normalize_profile(profile) for profile in CANDIDATE_PROFILES]
pd.DataFrame(asdict(profile) for profile in normalized_profiles)

## 6. Write Unit Tests for Default Behavior

These are notebook-local assertions. They are here to prevent accidental benchmark drift while iterating on candidate defaults.

In [ ]:
def run_notebook_assertions() -> None:
    legacy = normalize_profile(LEGACY_MODEL_PROFILE)
    assert legacy.chunk_size_bp == 20_000_000
    assert legacy.max_workers == 1
    assert legacy.window_stride == 1

    current = normalize_profile(CURRENT_SHIPPED_PROFILE)
    assert current.chunk_size_bp == ZFNShardingConfig().chunk_size_bp
    assert current.max_workers == ZFNShardingConfig().max_workers
    assert COMPARISON_BASELINE_PROFILE.profile_id == CURRENT_SHIPPED_PROFILE.profile_id

    tuned = normalize_profile(CURRENT_AUTOTUNE_PROFILE)
    assert tuned.chunk_size_bp in {8_000_000, 12_000_000}
    assert tuned.max_workers >= 1

    direct = build_params(current, SYNTHETIC_FASTA)
    assert direct.search_space_reference is None
    assert direct.search_space_fasta == str(SYNTHETIC_FASTA)
    assert direct.half_site_constraints.window_stride == 1

    try:
        normalize_profile(ProfileSpec(profile_id="bad", chunk_size_bp=0, max_workers=1))
    except ValueError:
        pass
    else:
        raise AssertionError("Expected chunk_size_bp validation failure")

    try:
        normalize_profile(ProfileSpec(profile_id="bad_workers", chunk_size_bp=1, max_workers=0))
    except ValueError:
        pass
    else:
        raise AssertionError("Expected max_workers validation failure")


run_notebook_assertions()
print("Notebook assertions passed.")

## 7. Inspect Effective Configuration at Runtime

This section is staged so you see useful output quickly:

1. Build a broad load-layout matrix across all cases and all profiles, with per-combination progress printing.
2. Run exhaustive search benchmarks only for the cases marked as `benchmark` and a focused default profile subset.
3. Keep larger synthetic cases visible in the layout matrix, but opt-in for expensive exhaustive runs.

If you want more coverage later, widen `DEFAULT_BENCHMARK_PROFILE_IDS` or flip additional load cases to `benchmark=True`.

In [ ]:
from IPython.display import display


class PhaseTimingCapture(logging.Handler):
    """Capture the structured per-phase timing payload emitted by the searcher."""

    def __init__(self) -> None:
        super().__init__(level=logging.INFO)
        self.phase_payload: dict[str, float] = {}

    def emit(self, record: logging.LogRecord) -> None:
        message = record.getMessage()
        marker = "ZFN search phase timings:"
        if marker not in message:
            return
        payload = message.split(marker, maxsplit=1)[1].strip()
        decoded = json.loads(payload)
        if isinstance(decoded, dict):
            self.phase_payload = {
                key: float(value) for key, value in decoded.items() if isinstance(value, (int, float))
            }


def mute_loggers(logger_names: list[str]) -> list[tuple[logging.Logger, int, list[logging.Handler], bool]]:
    """Temporarily silence noisy loggers during direct in-process benchmarking."""
    saved: list[tuple[logging.Logger, int, list[logging.Handler], bool]] = []
    for logger_name in logger_names:
        logger_obj = logging.getLogger(logger_name)
        saved.append((logger_obj, logger_obj.level, list(logger_obj.handlers), logger_obj.propagate))
        logger_obj.handlers = []
        logger_obj.propagate = False
        logger_obj.setLevel(logging.CRITICAL)
    return saved


def restore_loggers(saved: list[tuple[logging.Logger, int, list[logging.Handler], bool]]) -> None:
    """Restore logger state captured by mute_loggers()."""
    for logger_obj, level, handlers, propagate in saved:
        logger_obj.handlers = handlers
        logger_obj.propagate = propagate
        logger_obj.setLevel(level)


def inspect_effective_runtime(profile: ProfileSpec, search_space_fasta: Path) -> dict[str, Any]:
    """Return shard layout and effective worker information without running a full search."""
    searcher = ExhaustiveZFNOffTargetSearcher()
    params = build_params(normalize_profile(profile), search_space_fasta)
    muted = mute_loggers(
        [
            "sirnaforge.zfn.search",
            "sirnaforge.data.transcriptome_manager",
            "sirnaforge.data.genome_manager",
        ]
    )
    try:
        fasta_path = searcher._resolve_search_space_fasta(params)
        chrom_sequences = searcher._load_fasta(fasta_path)
        shard_specs = searcher._build_shard_specs(chrom_sequences, params)
        worker_cap = searcher._recommended_worker_cap(chrom_sequences, params)
    finally:
        restore_loggers(muted)

    effective_workers = min(worker_cap, len(shard_specs)) if shard_specs else 1
    total_bp = sum(len(sequence) for sequence in chrom_sequences.values())
    largest_contig_bp = max((len(sequence) for sequence in chrom_sequences.values()), default=0)
    return {
        "profile_id": profile.profile_id,
        "requested_chunk_size_bp": params.sharding.chunk_size_bp,
        "requested_max_workers": params.sharding.max_workers,
        "effective_workers": effective_workers,
        "thread_cap_default": searcher._thread_worker_cap(),
        "shard_count": len(shard_specs),
        "contig_count": len(chrom_sequences),
        "total_bp": total_bp,
        "largest_contig_bp": largest_contig_bp,
        "window_stride": params.half_site_constraints.window_stride,
        "required_overlap_bp": required_overlap_bp(params),
        "configured_overlap_bp": params.sharding.overlap_bp,
        "search_space_fasta": str(search_space_fasta),
    }


def top_site_signature(site: Any | None) -> tuple[Any, ...] | None:
    """Build a compact signature for top-hit consistency checks across repeats."""
    if site is None:
        return None
    return (
        site.chrom,
        site.start_1based,
        site.end_1based,
        site.strand,
        site.orientation,
        site.spacer_len,
        round(float(site.score), 6),
    )


def benchmark_profile(
    profile: ProfileSpec, search_space_fasta: Path, repeats: int = DEFAULT_EXHAUSTIVE_REPEATS
) -> pd.DataFrame:
    """Run repeated exhaustive searches for one profile and one FASTA."""
    rows: list[dict[str, Any]] = []
    normalized = normalize_profile(profile)
    runtime_info = inspect_effective_runtime(normalized, search_space_fasta)

    print(
        f"  profile={normalized.profile_id} chunk={runtime_info['requested_chunk_size_bp']} "
        f"workers={runtime_info['requested_max_workers']} effective={runtime_info['effective_workers']} "
        f"shards={runtime_info['shard_count']}"
    )

    for repeat_idx in range(1, repeats + 1):
        print(f"    repeat {repeat_idx}/{repeats} starting")
        searcher = ExhaustiveZFNOffTargetSearcher()
        params = build_params(normalized, search_space_fasta)
        handler = PhaseTimingCapture()
        search_logger = logging.getLogger("sirnaforge.zfn.search")
        original_level = search_logger.level
        original_handlers = list(search_logger.handlers)
        original_propagate = search_logger.propagate
        muted = mute_loggers(
            [
                "sirnaforge.data.transcriptome_manager",
                "sirnaforge.data.genome_manager",
            ]
        )
        search_logger.handlers = []
        search_logger.propagate = False
        search_logger.addHandler(handler)
        search_logger.setLevel(logging.INFO)

        started_at = time.perf_counter()
        try:
            ranked = searcher.search(params=params, annotation=None)
        finally:
            elapsed_s = time.perf_counter() - started_at
            search_logger.handlers = original_handlers
            search_logger.propagate = original_propagate
            search_logger.setLevel(original_level)
            restore_loggers(muted)

        print(f"    repeat {repeat_idx}/{repeats} finished in {elapsed_s:.2f}s with {len(ranked)} site(s)")
        top_site = ranked[0] if ranked else None
        rows.append(
            {
                "profile_id": normalized.profile_id,
                "repeat": repeat_idx,
                "elapsed_s": elapsed_s,
                "site_count": len(ranked),
                "top_site_signature": json.dumps(top_site_signature(top_site), default=str),
                "effective_workers": runtime_info["effective_workers"],
                "requested_max_workers": runtime_info["requested_max_workers"],
                "requested_chunk_size_bp": runtime_info["requested_chunk_size_bp"],
                "shard_count": runtime_info["shard_count"],
                "contig_count": runtime_info["contig_count"],
                "total_bp": runtime_info["total_bp"],
                "largest_contig_bp": runtime_info["largest_contig_bp"],
                "window_stride": runtime_info["window_stride"],
                "required_overlap_bp": runtime_info["required_overlap_bp"],
                "configured_overlap_bp": runtime_info["configured_overlap_bp"],
                "search_space_fasta": runtime_info["search_space_fasta"],
                "search_shards_s": handler.phase_payload.get("search_shards_s"),
                "dedupe_s": handler.phase_payload.get("dedupe_s"),
                "rank_s": handler.phase_payload.get("rank_s"),
                "resolve_inputs_s": handler.phase_payload.get("resolve_inputs_s"),
                "load_fasta_s": handler.phase_payload.get("load_fasta_s"),
                "build_shards_s": handler.phase_payload.get("build_shards_s"),
                "notes": normalized.notes,
            }
        )

    return pd.DataFrame(rows)


def benchmark_profiles(
    profiles: list[ProfileSpec],
    search_space_fasta: Path,
    repeats: int = DEFAULT_EXHAUSTIVE_REPEATS,
) -> pd.DataFrame:
    """Run one benchmark matrix for a single load case."""
    frames = [
        benchmark_profile(profile, search_space_fasta=search_space_fasta, repeats=repeats) for profile in profiles
    ]
    return pd.concat(frames, ignore_index=True)


def relative_spread(values: list[float]) -> float:
    """Compute normalized min/max spread around the median."""
    if len(values) <= 1:
        return 0.0
    median = statistics.median(values)
    if median == 0:
        return 0.0
    return (max(values) - min(values)) / median


def build_load_layout_matrix(profiles: list[ProfileSpec], load_cases: list[dict[str, Any]]) -> pd.DataFrame:
    """Inspect shard topology across many load cases without paying full search cost."""
    rows: list[dict[str, Any]] = []
    total_steps = len(load_cases) * len(profiles)
    step = 0
    for case in load_cases:
        for profile in profiles:
            step += 1
            print(f"[layout {step}/{total_steps}] case={case['name']} profile={profile.profile_id}")
            runtime_info = inspect_effective_runtime(profile, case["fasta"])
            rows.append(
                {
                    "case_name": case["name"],
                    "benchmark": case["benchmark"],
                    "profile_id": profile.profile_id,
                    **runtime_info,
                }
            )
    return pd.DataFrame(rows)


def summarize_runs(
    results: pd.DataFrame, baseline_profile_id: str = COMPARISON_BASELINE_PROFILE.profile_id
) -> pd.DataFrame:
    """Aggregate repeated runs into one row per case/profile combination."""
    if results.empty:
        raise ValueError("No benchmark rows were produced")

    summary_rows: list[dict[str, Any]] = []
    grouped = results.groupby(["case_name", "profile_id"], sort=False)
    for (case_name, profile_id), frame in grouped:
        elapsed_values = frame["elapsed_s"].tolist()
        median_elapsed_s = statistics.median(elapsed_values)
        summary_rows.append(
            {
                "case_name": case_name,
                "profile_id": profile_id,
                "median_elapsed_s": median_elapsed_s,
                "mean_elapsed_s": statistics.mean(elapsed_values),
                "min_elapsed_s": min(elapsed_values),
                "max_elapsed_s": max(elapsed_values),
                "relative_spread": relative_spread(elapsed_values),
                "site_count": int(frame["site_count"].iloc[0]),
                "top_site_consistent": frame["top_site_signature"].nunique(dropna=False) == 1,
                "effective_workers": int(frame["effective_workers"].iloc[0]),
                "requested_max_workers": int(frame["requested_max_workers"].iloc[0]),
                "requested_chunk_size_bp": int(frame["requested_chunk_size_bp"].iloc[0]),
                "shard_count": int(frame["shard_count"].iloc[0]),
                "window_stride": int(frame["window_stride"].iloc[0]),
                "benchmark_repeats": len(frame),
                "notes": frame["notes"].iloc[0],
            }
        )

    summary = pd.DataFrame(summary_rows)
    baseline_rows = summary.loc[summary["profile_id"] == baseline_profile_id, ["case_name", "median_elapsed_s"]].rename(
        columns={"median_elapsed_s": "baseline_median_elapsed_s"}
    )
    if baseline_rows.empty:
        raise ValueError(f"Baseline profile {baseline_profile_id!r} did not appear in the benchmark results")

    summary = summary.merge(baseline_rows, on="case_name", how="left")
    summary["speedup_vs_baseline"] = summary["baseline_median_elapsed_s"] / summary["median_elapsed_s"]
    return summary.sort_values(["case_name", "median_elapsed_s", "profile_id"]).reset_index(drop=True)


def select_recommendation(
    summary: pd.DataFrame,
    thresholds: ThresholdSpec = ThresholdSpec(),
    baseline_profile_id: str = COMPARISON_BASELINE_PROFILE.profile_id,
) -> pd.DataFrame:
    """Flag candidate profiles that look shippable under the default benchmark subset."""
    eligible = summary.copy()
    eligible = eligible.loc[eligible["window_stride"] == 1].copy()
    eligible["meets_speedup"] = eligible["speedup_vs_baseline"] >= 1.0 + thresholds.min_speedup_vs_baseline
    eligible["meets_stability"] = eligible["relative_spread"] <= thresholds.max_relative_spread
    eligible["meets_consistency"] = eligible["top_site_consistent"]
    eligible["is_baseline"] = eligible["profile_id"] == baseline_profile_id
    eligible["recommended"] = (
        eligible["meets_speedup"]
        & eligible["meets_stability"]
        & eligible["meets_consistency"]
        & ~eligible["is_baseline"]
    )
    return eligible.sort_values(
        ["case_name", "recommended", "median_elapsed_s"], ascending=[True, False, True]
    ).reset_index(drop=True)


def write_artifacts(
    layout_matrix: pd.DataFrame, results: pd.DataFrame, summary: pd.DataFrame, recommendations: pd.DataFrame
) -> None:
    """Persist benchmark artifacts for later review."""
    layout_matrix.to_csv(ARTIFACT_ROOT / "layout_matrix.csv", index=False)
    results.to_csv(ARTIFACT_ROOT / "benchmark_results.csv", index=False)
    summary.to_csv(ARTIFACT_ROOT / "benchmark_summary.csv", index=False)
    recommendations.to_csv(ARTIFACT_ROOT / "recommendations.csv", index=False)


def summarize_benchmark_suite(summary: pd.DataFrame, recommendations: pd.DataFrame) -> pd.DataFrame:
    """Produce one compact row per case for the quickest readout."""
    best_rows = (
        summary.sort_values(["case_name", "median_elapsed_s", "profile_id"])
        .groupby("case_name", as_index=False)
        .first()
    )
    recommended_rows = recommendations.loc[recommendations["recommended"]].copy()
    if not recommended_rows.empty:
        recommended_rows = recommended_rows.groupby("case_name", as_index=False).first()[
            ["case_name", "profile_id", "speedup_vs_baseline"]
        ]
        recommended_rows = recommended_rows.rename(
            columns={
                "profile_id": "recommended_profile_id",
                "speedup_vs_baseline": "recommended_speedup_vs_baseline",
            }
        )
        best_rows = best_rows.merge(recommended_rows, on="case_name", how="left")
    else:
        best_rows["recommended_profile_id"] = None
        best_rows["recommended_speedup_vs_baseline"] = None
    return best_rows


LOAD_CASES = [
    {"name": "smoke_single_contig", "fasta": SYNTHETIC_FASTA, "benchmark": True},
    {"name": "small_single_9mb", "fasta": SMALL_SINGLE_FASTA, "benchmark": True},
    {"name": "dual_contig_24mb", "fasta": DUAL_CONTIG_FASTA, "benchmark": True},
    {"name": "skewed_contigs_46mb", "fasta": SKEWED_CONTIG_FASTA, "benchmark": False},
    {"name": "sharded_single_48mb", "fasta": SHARDED_TUNING_FASTA, "benchmark": True},
    {"name": "diverse_multi_contig_72mb", "fasta": DIVERSE_TUNING_FASTA, "benchmark": False},
]

BENCHMARK_CASES = [case for case in LOAD_CASES if case["benchmark"]]
BENCHMARK_PROFILES = [profile for profile in CANDIDATE_PROFILES if profile.profile_id in DEFAULT_BENCHMARK_PROFILE_IDS]

layout_matrix = build_load_layout_matrix(CANDIDATE_PROFILES, LOAD_CASES)
display(layout_matrix)

benchmark_frames: list[pd.DataFrame] = []
for case_idx, case in enumerate(BENCHMARK_CASES, start=1):
    print(f"[benchmark case {case_idx}/{len(BENCHMARK_CASES)}] {case['name']}")
    case_frame = benchmark_profiles(
        BENCHMARK_PROFILES,
        search_space_fasta=case["fasta"],
        repeats=DEFAULT_EXHAUSTIVE_REPEATS,
    ).assign(case_name=case["name"], benchmark=True)
    benchmark_frames.append(case_frame)

benchmark_results = pd.concat(benchmark_frames, ignore_index=True)
benchmark_summary = summarize_runs(benchmark_results)
recommendations = select_recommendation(benchmark_summary)
suite_summary = summarize_benchmark_suite(benchmark_summary, recommendations)

write_artifacts(layout_matrix, benchmark_results, benchmark_summary, recommendations)

print("Artifacts written:")
print(f"  {ARTIFACT_ROOT / 'layout_matrix.csv'}")
print(f"  {ARTIFACT_ROOT / 'benchmark_results.csv'}")
print(f"  {ARTIFACT_ROOT / 'benchmark_summary.csv'}")
print(f"  {ARTIFACT_ROOT / 'recommendations.csv'}")

display(benchmark_summary)
display(recommendations)
display(suite_summary)